In [3]:
from transformers import Qwen2VLProcessor
import textwrap
from huggingface_hub import HfApi
from transformers import AutoProcessor
import torch
from transformers import Qwen2VLConfig  
from transformers import Qwen2VLForConditionalGenerationWithAudio 
from transformers import Qwen2VLAudioConfig
from qwen_vl_utils import vision_process, process_vision_info, fetch_audio
from typing import List, Dict, Any, Union
import librosa
import io
from datasets import load_dataset


api = HfApi()

/Users/yalimdemirkesen/Desktop/LLM/speech_recognition/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Create the Processor

In [4]:
#TASK: format the data for the model by brining each training sample into the OpenAI conversation format

def format_data(sample):
    """
    Convert a training sample into OpenAI conversation format.
    Each sample contains audio data and its corresponding transcription.
    """
    # Extract the transcription text
    transcription = sample['text']
    
    # Create the conversation format
    conversation = [
        {
            "content":[{
                            "text": "You are an ASR model that transcribes speech to text. Avoid additional explanation unless absolutely necessary.",
                            "type": "text"
                        }],
            "role":"system"
        },
        {
            "content": [{
                    "audio": sample['wav']['bytes'],  # Raw audio bytes
                    "type": "audio"
                },
                {
                    "text": "Transcribe this speech into text.",
                    "type": "text",
                }
            ],
            "role": "user"
        },
        {
            "content": [
                {
                    "text": transcription,
                    "type": "text"
                }
            ],
            "role": "assistant",
        }
    ]
    
    return conversation

In [5]:
processor = Qwen2VLProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")
special_tokens_dict = {
    "additional_special_tokens": ["<|audio_start|>", "<|audio_pad|>", "<|audio_end|>"]
}
num_added = processor.tokenizer.add_special_tokens(special_tokens_dict)
print(f"Added {num_added} special tokens for audio")

Added 3 special tokens for audio


In [6]:
audio_token_ids = processor.tokenizer.convert_tokens_to_ids(["<|audio_start|>", "<|audio_pad|>", "<|audio_end|>"])
print(f"Audio token IDs: {audio_token_ids}")

Audio token IDs: [151657, 151658, 151659]


In [7]:
print(processor.tokenizer.chat_template)

{% set image_count = namespace(value=0) %}{% set video_count = namespace(value=0) %}{% for message in messages %}{% if loop.first and message['role'] != 'system' %}<|im_start|>system
You are a helpful assistant.<|im_end|>
{% endif %}<|im_start|>{{ message['role'] }}
{% if message['content'] is string %}{{ message['content'] }}<|im_end|>
{% else %}{% for content in message['content'] %}{% if content['type'] == 'image' or 'image' in content or 'image_url' in content %}{% set image_count.value = image_count.value + 1 %}{% if add_vision_id %}Picture {{ image_count.value }}: {% endif %}<|vision_start|><|image_pad|><|vision_end|>{% elif content['type'] == 'video' or 'video' in content %}{% set video_count.value = video_count.value + 1 %}{% if add_vision_id %}Video {{ video_count.value }}: {% endif %}<|vision_start|><|video_pad|><|vision_end|>{% elif 'text' in content %}{{ content['text'] }}{% endif %}{% endfor %}<|im_end|>
{% endif %}{% endfor %}{% if add_generation_prompt %}<|im_start|>assi

In [8]:
chat_template_w_audio = textwrap.dedent(r"""
{% set image_count = namespace(value=0) %}
{% set video_count = namespace(value=0) %}
{% set audio_count = namespace(value=0) %}
{% for message in messages %}
{% if loop.first and message['role'] != 'system' %}
<|im_start|>system
You are a helpful assistant.<|im_end|>
{% endif %}
<|im_start|>{{ message['role'] }}
{% if message['content'] is string %}
{{ message['content'] }}<|im_end|>
{% else %}
{% for content in message['content'] %}
    {% if content['type'] == 'image' or 'image' in content or 'image_url' in content %}
        {% set image_count.value = image_count.value + 1 %}
        {% if add_vision_id %}Picture {{ image_count.value }}: {% endif %}
        <|vision_start|><|image_pad|><|vision_end|>
    {% elif content['type'] == 'video' or 'video' in content %}
        {% set video_count.value = video_count.value + 1 %}
        {% if add_vision_id %}Video {{ video_count.value }}: {% endif %}
        <|vision_start|><|video_pad|><|vision_end|>
    {% elif content.get('type') in ['audio', 'input_audio'] or 'audio' in content %}
        {% set audio_count.value = audio_count.value + 1 %}
        {% if add_audio_id %}Audio {{ audio_count.value }}: {% endif %}
        <|audio_start|><|audio_pad|><|audio_end|>
    {% elif 'text' in content %}
        {{ content['text'] }}
    {% endif %}
{% endfor %}
<|im_end|>
{% endif %}
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{% endif %}
""").strip()

In [9]:
processor.chat_template = chat_template_w_audio

In [8]:
speech_brain = load_dataset("speechbrain/LargeScaleASR", "small",streaming=True)
sample = next(iter(speech_brain['train'].take(1)))

In [9]:
formatted_conversation = format_data(sample)

In [10]:
text_with_template = processor.apply_chat_template(
    formatted_conversation, 
    tokenize=False, 
    add_generation_prompt=False
)

In [11]:
print(text_with_template)

<|im_start|>system
        You are an ASR model that transcribes speech to text. Avoid additional explanation unless absolutely necessary.
<|im_end|>
<|im_start|>user
        <|audio_start|><|audio_pad|><|audio_end|>
        Transcribe this speech into text.
<|im_end|>
<|im_start|>assistant
        AND WHAT ABOUT INTEROPERABILITY IN THE RAIL SECTOR ARE NATIONAL BARRIERS PREVENTING PROGRESS IN THIS AREA AS WELL OR IS THERE AN UNWILLINGNESS ON THE PART OF THE RAIL INDUSTRY TO EMBRACE THE CONCEPT OF INTEROPERABILITY
<|im_end|>



In [12]:
audio_bytes = sample['wav']['bytes']
audio_data, sampling_rate = librosa.load(io.BytesIO(audio_bytes), sr=None)

In [13]:
audio_data

array([ 0.00186157,  0.00012207,  0.00149536, ..., -0.00494385,
       -0.00268555, -0.01150513], shape=(273920,), dtype=float32)

In [14]:
batch = processor(text=text_with_template, 
                  audios=audio_data, 
                  audio_sample_rate=sampling_rate, 
                  return_tensors="pt")

In [15]:
batch

{'input_ids': tensor([[151644,   8948,    198,    286,   1446,    525,    458,   5752,     49,
           1614,    429,   1356,  55136,   8806,    311,   1467,     13,  34006,
           5107,  16148,   7241,  10875,   5871,    624, 151645,    198, 151644,
            872,    198,    260, 151657, 151658, 151658, 151658, 151658, 151658,
         151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658,
         151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658,
         151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658,
         151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658,
         151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658,
         151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658,
         151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658,
         151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658, 151658,
         15165

In [16]:
print(text_with_template)

<|im_start|>system
        You are an ASR model that transcribes speech to text. Avoid additional explanation unless absolutely necessary.
<|im_end|>
<|im_start|>user
        <|audio_start|><|audio_pad|><|audio_end|>
        Transcribe this speech into text.
<|im_end|>
<|im_start|>assistant
        AND WHAT ABOUT INTEROPERABILITY IN THE RAIL SECTOR ARE NATIONAL BARRIERS PREVENTING PROGRESS IN THIS AREA AS WELL OR IS THERE AN UNWILLINGNESS ON THE PART OF THE RAIL INDUSTRY TO EMBRACE THE CONCEPT OF INTEROPERABILITY
<|im_end|>



In [17]:
audio_vals = batch["audio_values"]
audio_vals = audio_vals.unsqueeze(0)
audio_vals.shape

torch.Size([1, 273920])

In [18]:
int(batch["number_encoder_tokens"].item())

856

In [19]:
# save the processor
local_save_path = "/Users/yalimdemirkesen/Desktop/LLM/speech_recognition/hf/qwen_w_audio_processor"
processor.save_pretrained(local_save_path)
print(f"Saved processor to: {local_save_path}")

Saved processor to: /Users/yalimdemirkesen/Desktop/LLM/speech_recognition/hf/qwen_w_audio_processor


In [20]:
# load to huggingface hub
repo_name = "mdmy/whisper_asr_finetuning"

api.upload_folder(
    folder_path="/Users/yalimdemirkesen/Desktop/LLM/speech_recognition/hf/",
    repo_id=repo_name,
    commit_message="Add audio-enabled Qwen processor/tokenizer"
)

Processing Files (9 / 9): 100%|██████████| 35.7GB / 35.7GB, 3.70MB/s  
New Data Upload: 100%|██████████|  178MB /  178MB, 3.70MB/s  


CommitInfo(commit_url='https://huggingface.co/mdmy/whisper_asr_finetuning/commit/f5e4e57647e98b8e677f0116079c51a9ba9fff89', commit_message='Add audio-enabled Qwen processor/tokenizer', commit_description='', oid='f5e4e57647e98b8e677f0116079c51a9ba9fff89', pr_url=None, repo_url=RepoUrl('https://huggingface.co/mdmy/whisper_asr_finetuning', endpoint='https://huggingface.co', repo_type='model', repo_id='mdmy/whisper_asr_finetuning'), pr_revision=None, pr_num=None)

# Test on Model

In [10]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
BASE_ID = "Qwen/Qwen2-VL-7B-Instruct"


processor = AutoProcessor.from_pretrained(
    "mdmy/whisper_asr_finetuning",
    subfolder="qwen_w_audio_processor",
    trust_remote_code=True
)

## Audio

In [11]:
speech_brain = load_dataset("speechbrain/LargeScaleASR", "small",streaming=True)
sample = next(iter(speech_brain['train'].take(1)))

In [12]:
audio_bytes = sample['wav']['bytes']
audio_data, sampling_rate = vision_process.fetch_audio(audio_bytes)

In [13]:
audio_data

array([ 0.00186157,  0.00012207,  0.00149536, ..., -0.00494385,
       -0.00268555, -0.01150513], shape=(273920,))

In [14]:
cfg = Qwen2VLConfig.from_pretrained(BASE_ID, trust_remote_code=True)

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


In [15]:
cfg

Qwen2VLConfig {
  "architectures": [
    "Qwen2VLForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "audio_config": {
    "model_type": "qwen2_vl"
  },
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "image_token_id": 151655,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "max_position_embeddings": 32768,
  "max_window_layers": 28,
  "model_type": "qwen2_vl",
  "num_attention_heads": 28,
  "num_hidden_layers": 28,
  "num_key_value_heads": 4,
  "rms_norm_eps": 1e-06,
  "rope_scaling": {
    "mrope_section": [
      16,
      24,
      24
    ],
    "rope_type": "default",
    "type": "default"
  },
  "rope_theta": 1000000.0,
  "sliding_window": 32768,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.47.0",
  "use_audio": false,
  "use_cache": true,
  "use_sliding_window": false,
  "video_token_id": 151656,
  "vision_config": {
    "in_chans": 3,
    "model_type": "qwen2_

In [16]:
cfg.use_audio = True
audio_cfg = Qwen2VLAudioConfig()
cfg.audio_config = audio_cfg

In [17]:
cfg

Qwen2VLConfig {
  "architectures": [
    "Qwen2VLForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "audio_config": {
    "model_type": "qwen2_vl"
  },
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "image_token_id": 151655,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "max_position_embeddings": 32768,
  "max_window_layers": 28,
  "model_type": "qwen2_vl",
  "num_attention_heads": 28,
  "num_hidden_layers": 28,
  "num_key_value_heads": 4,
  "rms_norm_eps": 1e-06,
  "rope_scaling": {
    "mrope_section": [
      16,
      24,
      24
    ],
    "rope_type": "default",
    "type": "default"
  },
  "rope_theta": 1000000.0,
  "sliding_window": 32768,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.47.0",
  "use_audio": true,
  "use_cache": true,
  "use_sliding_window": false,
  "video_token_id": 151656,
  "vision_config": {
    "in_chans": 3,
    "model_type": "qwen2_v

In [18]:
model = Qwen2VLForConditionalGenerationWithAudio.from_pretrained(
    BASE_ID,
    config=cfg,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    ignore_mismatched_sizes=True,   # audio heads are new
    trust_remote_code=True,
    device_map=None,                # single device for the test
).to(DEVICE)
model.eval()

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46
Loading checkpoint shards: 100%|██████████| 5/5 [00:21<00:00,  4.23s/it]
Some weights of Qwen2VLForConditionalGenerationWithAudio were not initialized from the model checkpoint at Qwen/Qwen2-VL-7B-Instruct and are newly initialized: ['audio_module.audio_projection.weight', 'audio_module.whisper_encoder.conv1.bias', 'audio_module.whisper_encoder.conv1.weight', 'audio_module.whisper_encoder.conv2.bias', 'audio_module.whisper_encoder.conv2.weight', 'audio_module.whisper_encoder.embed_positions.weight', 'audio_module.whisper_encoder.layer_norm.bias', 'audio_module.whisper_encoder.layer_norm.weight', 'audio_module.whisper_encoder.layers.0.fc1.bias', 'audio_module.whisper_encoder.layers.0.fc1.weight', 'audio_module.whisper_encoder.l

Qwen2VLForConditionalGenerationWithAudio(
  (visual): Qwen2VisionTransformerPretrainedModel(
    (patch_embed): PatchEmbed(
      (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
    )
    (rotary_pos_emb): VisionRotaryEmbedding()
    (blocks): ModuleList(
      (0-31): 32 x Qwen2VLVisionBlock(
        (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (attn): VisionSdpaAttention(
          (qkv): Linear(in_features=1280, out_features=3840, bias=True)
          (proj): Linear(in_features=1280, out_features=1280, bias=True)
        )
        (mlp): VisionMlp(
          (fc1): Linear(in_features=1280, out_features=5120, bias=True)
          (act): QuickGELUActivation()
          (fc2): Linear(in_features=5120, out_features=1280, bias=True)
        )
      )
    )
    (merger): PatchMerger(
      (ln_q): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
      (

In [19]:
def format_data(sample):
    """
    Convert a training sample into OpenAI conversation format.
    Each sample contains audio data and its corresponding transcription.
    """
    # Extract the transcription text
    transcription = sample['text']
    
    # Create the conversation format
    conversation = [
        {
            "content":[{
                            "text": "You are an ASR model that transcribes speech to text. Avoid additional explanation unless absolutely necessary.",
                            "type": "text"
                        }],
            "role":"system"
        },
        {
            "content": [{
                    "audio": sample['wav']['bytes'],  # Raw audio bytes
                    "type": "audio"
                },
                {
                    "text": "Transcribe this speech into text.",
                    "type": "text",
                }
            ],
            "role": "user"
        },
        {
            "content": [
                {
                    "text": transcription,
                    "type": "text"
                }
            ],
            "role": "assistant",
        }
    ]
    
    return conversation

In [20]:
def format_data_for_inference(sample):
    """
    Convert a sample into OpenAI conversation format for inference.
    Does NOT include the transcription - that's what the model should generate.
    """
    # Create the conversation format for inference
    conversation = [
        {
            "content": [{
                "text": "You are an ASR model that transcribes speech to text. Avoid additional explanation unless absolutely necessary.",
                "type": "text"
            }],
            "role": "system"
        },
        {
            "content": [{
                "audio": sample['wav']['bytes'],  # Raw audio bytes
                "type": "audio"
            },
            {
                "text": "Transcribe this speech into text.",
                "type": "text",
            }],
            "role": "user"
        }
    ]
    
    return conversation

In [21]:
def transcribe_audio(audio_sample: Dict,
                     processor,
                     model,
                     max_new_tokens: int = 256,
                     temperature: float = 0.0,
                     device: Union[str, torch.device] = "cpu",
                     ) -> str:
    """
    - convert to conversation format
    - convert to chat template
    - apply processing
    - apply the model
    - decode the generated tokens into output text
    """

    # convert to conversation format (inference only - no target transcription)
    formatted_conversation = format_data_for_inference(audio_sample)  

    # convert to chat template
    text_with_template = processor.apply_chat_template(formatted_conversation, 
                                                        tokenize=False, 
                                                        add_generation_prompt=True) 

    audio_bytes = audio_sample['wav']['bytes'] 
    audio_array, sampling_rate = fetch_audio(audio_bytes)
    
    # apply processing
    batch = processor(text=text_with_template, 
                  audios=audio_array, 
                  audio_sample_rate=sampling_rate, 
                  return_tensors="pt")
    
    # apply the model
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    audio_vals = batch["audio_values"]
    audio_vals = audio_vals.unsqueeze(0)
    audio_values = audio_vals.to(device).to(torch.float32)

    model.eval()
    with torch.inference_mode():
        gen_out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            audio_values=audio_values,  # consumed on step 0 by prepare_inputs_for_generation
            max_new_tokens=max_new_tokens,  # Use the parameter
            do_sample=False,            # set True + temperature/top_p if you want sampling
            use_cache=True,
            return_dict_in_generate=True,
            # output_scores=False,        # flip on if you need token-level scores
        )
        generated_ids = gen_out.sequences

    # decode the generated tokens into output text - only the newly generated part
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(batch["input_ids"], generated_ids)
    ]
    
    text = processor.tokenizer.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    return text

In [22]:
formatted_conversation = format_data_for_inference(sample)

In [23]:
formatted_conversation

[{'content': [{'text': 'You are an ASR model that transcribes speech to text. Avoid additional explanation unless absolutely necessary.',
    'type': 'text'}],
  'role': 'system'},
 {'content': [{'audio': b'RIFFF\\\x08\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x80>\x00\x00\x00}\x00\x00\x02\x00\x10\x00LIST\x1a\x00\x00\x00INFOISFT\r\x00\x00\x00Lavf61.1.100\x00\x00data\x00\\\x08\x00=\x00\x04\x001\x00\xe8\xff\t\x009\x00\xda\xff\xcc\xff\x92\xff\x89\xff\xa3\xff\xc5\xff\xfc\xff\xcc\xff\xb1\xff\x9a\xff\xb6\xff\xb5\xff\x95\xff\xe0\xff\x13\x00\x1a\x00\x15\x00\xf8\xff\xfb\xff\x0b\x00:\x00S\x00<\x00\x1d\x00\xe7\xff\xb5\xff\xa3\xff\xa5\xff\xbb\xff\xa2\xff\x80\xff\x93\xff\xbe\xff\xbf\xff\x85\xffj\xff{\xff\xcb\xff\xf6\xff\xd8\xff\xe1\xff\xb9\xff\x9a\xff\xcd\xff\xf3\xff\x06\x00\xf3\xff\x07\x00\x11\x00\xf1\xff\xdf\xff\xdd\xff\xe1\xff\xce\xff\x91\xff\xa7\xff\xe0\xff\x95\xff\xb2\xff\xfd\xff\xda\xff\xf4\xff\xf0\xff\x01\x00\xed\xff\xc6\xff\x14\x00\xe9\xff\xc2\xff\xd5\xff\xad\xff\x89\xffs\xffs\xff{\xffb\x

In [24]:
text_with_template = processor.apply_chat_template(formatted_conversation, 
                                                        tokenize=False, 
                                                        add_generation_prompt=True) 

In [25]:
print(text_with_template)

<|im_start|>system
        You are an ASR model that transcribes speech to text. Avoid additional explanation unless absolutely necessary.
<|im_end|>
<|im_start|>user
        <|audio_start|><|audio_pad|><|audio_end|>
        Transcribe this speech into text.
<|im_end|>
<|im_start|>assistant



In [26]:
audio_bytes = sample['wav']['bytes'] 
audio_array, sampling_rate = fetch_audio(audio_bytes)

In [27]:
batch = processor(text=text_with_template, 
                  audios=audio_array, 
                  audio_sample_rate=sampling_rate, 
                  return_tensors="pt")

In [28]:
input_ids = batch["input_ids"].to(DEVICE)
attention_mask = batch["attention_mask"].to(DEVICE)

In [29]:
audio_vals = batch["audio_values"]
audio_vals = audio_vals.unsqueeze(0)
audio_values = audio_vals.to(DEVICE).to(torch.float32)

In [30]:
model.eval()
with torch.inference_mode():
    gen_out = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        audio_values=audio_values,  # consumed on step 0 by prepare_inputs_for_generation
        max_new_tokens=128,
        do_sample=False,            # set True + temperature/top_p if you want sampling
        use_cache=True,
        return_dict_in_generate=True,
        # output_scores=False,        # flip on if you need token-level scores
    )
    generated_ids = gen_out.sequences


/Users/yalimdemirkesen/Desktop/LLM/speech_recognition/transformers/src/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/Users/yalimdemirkesen/Desktop/LLM/speech_recognition/transformers/src/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.001` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/Users/yalimdemirkesen/Desktop/LLM/speech_recognition/transformers/src/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
From v

In [31]:
# decode the generated tokens into output text
generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(batch["input_ids"], generated_ids)
    ]

In [32]:
text = processor.tokenizer.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

In [33]:
text

"I'm sorry, but the speech you provided is not transcribable as it contains a mix of letters, numbers, and symbols that do not form a coherent sentence or message. Please provide a clear and readable speech for transcription."

# Test on Model for Vision

In [ ]:
model = Qwen2VLForConditionalGenerationWithAudio.from_pretrained(
    BASE_ID,
    # config=cfg,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    ignore_mismatched_sizes=True,   # audio heads are new
    trust_remote_code=True,
    device_map=None,                # single device for the test
).to(DEVICE)
model.eval()


In [34]:
url = "https://t4.ftcdn.net/jpg/01/57/82/05/360_F_157820583_agejYX5XeczPZuWRSCDF2YYeCGwJqUdG.jpg"

GENERAL_MM_SYSTEM = (
    "You are a helpful, general-purpose multimodal assistant. "
    "You can understand text, images, and audio. "
    "Follow the user's instructions precisely. "
    "For detection-style requests, be concise and return coordinates if asked."
)

message = [
            {"role": "system", "content": [{"type": "text", "text": GENERAL_MM_SYSTEM}]},
                {
                        "role": "user",
                        "content": [
                            {"type": "image", "image": f"{url}"},
                            {"type": "text", "text": "Detect the bounding box of the red car."},
            ],
        }
    ]

In [35]:
text_default = processor.apply_chat_template(
    message, tokenize=False, add_generation_prompt=True
)

In [36]:
image_inputs_default, video_inputs_default = process_vision_info(message)

In [37]:
inputs_default = processor(
    text=[text_default],
    images=image_inputs_default,
    videos=video_inputs_default,
    padding=True,
    return_tensors="pt",
).to(DEVICE)

In [38]:
model.eval()
with torch.inference_mode():
    print("Starting generation with custom model + custom processor for image task...")
    gen_custom = model.generate(
        **inputs_default,
        max_new_tokens=256,
        use_cache=True,
        return_dict_in_generate=True,
    )

    generated_ids_trimmed_custom = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs_default.input_ids, gen_custom.sequences)
]

Starting generation with custom model + custom processor for image task...


In [39]:
output_text_custom = processor.batch_decode(
    generated_ids_trimmed_custom, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

In [40]:
print(output_text_custom[0])

red car(359,569),(493,949)


In [42]:
# TASK: Load the Qwen2VL checkpoint into your model, load whisper turbo, save and push to your repo -  Now, the model and tokenizer should both be there.
model_save_path = "/Users/yalimdemirkesen/Desktop/LLM/speech_recognition/hf/qwen2vl_with_audio_asr"
model.save_pretrained(model_save_path)
print(f"Saved model to: {model_save_path}")


Saved model to: /Users/yalimdemirkesen/Desktop/LLM/speech_recognition/hf/qwen2vl_with_audio_asr


In [ ]:
# save audio_config to local 
# /Users/yalimdemirkesen/Desktop/LLM/speech_recognition/hf/model_config.json
audio_cfg.save_pretrained("/Users/yalimdemirkesen/Desktop/LLM/speech_recognition/hf/model_config/")

In [4]:
# push the model to HF hub
repo_name = "mdmy/whisper_asr_finetuning"

api.upload_folder(
    folder_path="/Users/yalimdemirkesen/Desktop/LLM/speech_recognition/hf/",
    repo_id=repo_name,
    commit_message="Add Qwen2VL model with audio encoder for ASR"
)

Processing Files (9 / 9): 100%|██████████| 35.7GB / 35.7GB, 3.75GB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


CommitInfo(commit_url='https://huggingface.co/mdmy/whisper_asr_finetuning/commit/a3e0f554a89719557b3b1c6d1d61ffa309837c1f', commit_message='Add Qwen2VL model with audio encoder for ASR', commit_description='', oid='a3e0f554a89719557b3b1c6d1d61ffa309837c1f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/mdmy/whisper_asr_finetuning', endpoint='https://huggingface.co', repo_type='model', repo_id='mdmy/whisper_asr_finetuning'), pr_revision=None, pr_num=None)

In [ ]:
# test reloading the model from HF repo
model = Qwen2VLForConditionalGenerationWithAudio.from_pretrained(
    "mdmy/whisper_asr_finetuning",
    subfolder="qwen2vl_with_audio_asr",
    trust_remote_code=True
)
